# Baseline and Pretrained model Comparison

This notebook follows two stages:

1. Main comparison (5 models together):
   - VADER baseline
   - BERT-5class (mapped to 3 classes)
   - BERT-3class (BERT-based)
   - BERTweet-3class (RoBERTa-based encoder)
   - RoBERTa-3class (RoBERTa-based)
2. Separate reference experiment:
   - RoBERTa-base without task-specific training (for Person3 fine-tuning context)

All reported metrics include: Accuracy, Macro-F1, Macro-Precision, Macro-Recall, and class-wise F1.

## Phase A - Setup

In [1]:
from pathlib import Path
from typing import Dict, List

import pandas as pd
from sklearn.metrics import accuracy_score, classification_report, f1_score, precision_score, recall_score
from tqdm import tqdm
from transformers import AutoModelForSequenceClassification, AutoTokenizer, pipeline
from vaderSentiment.vaderSentiment import SentimentIntensityAnalyzer

id2label = {0: "negative", 1: "neutral", 2: "positive"}
label_order = [0, 1, 2]

model_vader = "VADER (baseline)"
model_bert5 = "BERT-5class (pretrained, direct call)"
model_bert3 = "BERT-3class (pretrained, direct call)"
model_bertweet3 = "BERTweet-3class (pretrained, direct call)"
model_roberta3 = "RoBERTa-3class (pretrained, direct call)"
model_roberta_base_ref = "RoBERTa-base (no fine-tuning, reference)"

bert5_checkpoint = "nlptown/bert-base-multilingual-uncased-sentiment"
bert3_checkpoint = "Priyanka-Balivada/bert-5-epoch-sentiment"
bertweet3_checkpoint = "cardiffnlp/bertweet-base-sentiment"
roberta3_checkpoint = "cardiffnlp/twitter-roberta-base-sentiment-latest"
roberta_base_checkpoint = "roberta-base"


In [2]:
def map_model_label_to_id(raw_label: str) -> int:
    s = raw_label.strip().lower()
    if s in {"label_0", "label_1", "label_2"}:
        return int(s[-1])
    if "neg" in s:
        return 0
    if "neu" in s:
        return 1
    if "pos" in s:
        return 2
    raise ValueError(f"Unable to parse model label: {raw_label}")


def map_bert_stars_to_id(stars_label: str) -> int:
    stars = int(stars_label.strip().split()[0])
    if stars <= 2:
        return 0
    if stars == 3:
        return 1
    return 2

## Phase B - VADER Threshold Selection on Validation Set

In [3]:
def predict_vader(texts: List[str], threshold: float = 0.05) -> List[int]:
    analyzer = SentimentIntensityAnalyzer()
    preds = []
    for text in texts:
        compound = analyzer.polarity_scores(text)["compound"]
        if compound >= threshold:
            preds.append(2)
        elif compound <= -threshold:
            preds.append(0)
        else:
            preds.append(1)
    return preds

In [4]:
# Run right after predict_vader(): VADER threshold selection on ALL validation sets
val_paths = [
    Path("datasets/tweeteval_sentiment_3class_val.csv"),
    Path("datasets/sst_3class_val.csv")
]

threshold_candidates = [0.05, 0.08]
rows = []

for val_path in val_paths:
    val_df = pd.read_csv(val_path).dropna(subset=["text", "label_id"]).copy()
    texts_val = val_df["text"].astype(str).tolist()
    y_val = val_df["label_id"].astype(int).tolist()

    for thr in threshold_candidates:
        preds = predict_vader(texts_val, threshold=thr)
        rows.append({
            "threshold": thr,
            "dataset": val_path.name,
            "model": model_vader,
            "accuracy": accuracy_score(y_val, preds),
            "f1_macro": f1_score(y_val, preds, average="macro", labels=label_order, zero_division=0),
            "precision_macro": precision_score(y_val, preds, average="macro", labels=label_order, zero_division=0),
            "recall_macro": recall_score(y_val, preds, average="macro", labels=label_order, zero_division=0)
        })

vader_threshold_result = pd.DataFrame(rows)
vader_threshold_result.sort_values(["dataset", "f1_macro"], ascending=[True, False])

,threshold,dataset,model,accuracy,f1_macro,precision_macro,recall_macro
3,0.08,sst_3class_val.csv,VADER (baseline),0.511392,0.468258,0.479942,0.473755
2,0.05,sst_3class_val.csv,VADER (baseline),0.516456,0.468141,0.477479,0.473070
1,0.08,tweeteval_sentiment_3class_val.csv,VADER (baseline),0.547412,0.539955,0.544365,0.562464
0,0.05,tweeteval_sentiment_3class_val.csv,VADER (baseline),0.543740,0.536542,0.545080,0.563873


Based on validation results, we select a VADER threshold of ±0.08, which gives better performance.

## Phase C - Inference Functions

In [6]:
def predict_bert5(texts: List[str], batch_size: int = 64) -> List[int]:
    tokenizer = AutoTokenizer.from_pretrained(bert5_checkpoint)
    model = AutoModelForSequenceClassification.from_pretrained(bert5_checkpoint)
    classifier = pipeline("text-classification", model=model, tokenizer=tokenizer)

    preds = []
    for i in tqdm(range(0, len(texts), batch_size), desc="BERT-5class inference"):
        batch = texts[i : i + batch_size]
        outputs = classifier(batch, truncation=True, max_length=512)
        preds.extend(map_bert_stars_to_id(out["label"]) for out in outputs)
    return preds


def predict_bert3(texts: List[str], batch_size: int = 64) -> List[int]:
    tokenizer = AutoTokenizer.from_pretrained(bert3_checkpoint)
    model = AutoModelForSequenceClassification.from_pretrained(bert3_checkpoint)
    classifier = pipeline("text-classification", model=model, tokenizer=tokenizer)

    preds = []
    for i in tqdm(range(0, len(texts), batch_size), desc="BERT-3class inference"):
        batch = texts[i : i + batch_size]
        outputs = classifier(batch, truncation=True, max_length=512)
        preds.extend(map_model_label_to_id(out["label"]) for out in outputs)
    return preds


def predict_bertweet3(texts: List[str], batch_size: int = 64) -> List[int]:
    tokenizer = AutoTokenizer.from_pretrained(bertweet3_checkpoint)
    model = AutoModelForSequenceClassification.from_pretrained(bertweet3_checkpoint)
    classifier = pipeline("text-classification", model=model, tokenizer=tokenizer)

    preds = []
    for i in tqdm(range(0, len(texts), batch_size), desc="BERTweet-3class inference"):
        batch = texts[i : i + batch_size]
        outputs = classifier(batch, truncation=True, max_length=512)
        preds.extend(map_model_label_to_id(out["label"]) for out in outputs)
    return preds


def predict_roberta3(texts: List[str], batch_size: int = 64) -> List[int]:
    tokenizer = AutoTokenizer.from_pretrained(roberta3_checkpoint)
    model = AutoModelForSequenceClassification.from_pretrained(roberta3_checkpoint)
    classifier = pipeline("text-classification", model=model, tokenizer=tokenizer)

    preds = []
    for i in tqdm(range(0, len(texts), batch_size), desc="RoBERTa-3class inference"):
        batch = texts[i : i + batch_size]
        outputs = classifier(batch, truncation=True, max_length=512)
        preds.extend(map_model_label_to_id(out["label"]) for out in outputs)
    return preds


 # Non-task-trained reference
def predict_roberta_base_reference(texts: List[str], batch_size: int = 64) -> List[int]:
    tokenizer = AutoTokenizer.from_pretrained(roberta_base_checkpoint)
    model = AutoModelForSequenceClassification.from_pretrained(
        roberta_base_checkpoint,
        num_labels=3,
        ignore_mismatched_sizes=True,
    )
    classifier = pipeline("text-classification", model=model, tokenizer=tokenizer)

    preds = []
    for i in tqdm(range(0, len(texts), batch_size), desc="RoBERTa-base reference"):
        batch = texts[i : i + batch_size]
        outputs = classifier(batch, truncation=True, max_length=512)
        preds.extend(map_model_label_to_id(out["label"]) for out in outputs)
    return preds

## Phase D - Evaluation Utilities

In [7]:

def compute_metrics(y_true: List[int], y_pred: List[int]) -> Dict[str, float]:
    report = classification_report(
        y_true,
        y_pred,
        labels=label_order,
        target_names=[id2label[i] for i in label_order],
        output_dict=True,
        zero_division=0,
    )
    return {
        "accuracy": accuracy_score(y_true, y_pred),
        "f1_macro": f1_score(y_true, y_pred, average="macro", labels=label_order, zero_division=0),
        "precision_macro": precision_score(y_true, y_pred, average="macro", labels=label_order, zero_division=0),
        "recall_macro": recall_score(y_true, y_pred, average="macro", labels=label_order, zero_division=0),
        "f1_negative": report["negative"]["f1-score"],
        "f1_neutral": report["neutral"]["f1-score"],
        "f1_positive": report["positive"]["f1-score"],
    }


def evaluate_main_models(
    df: pd.DataFrame,
    dataset_name: str,
    batch_size: int = 64,
    vader_threshold: float = 0.08,
) -> pd.DataFrame:
    df = df.dropna(subset=["text", "label_id"]).copy()
    texts = df["text"].astype(str).tolist()
    labels = df["label_id"].astype(int).tolist()

    rows = []

    vader_preds = predict_vader(texts, threshold=vader_threshold)
    m = compute_metrics(labels, vader_preds)
    m.update({"dataset": dataset_name, "model": model_vader})
    rows.append(m)

    bert5_preds = predict_bert5(texts, batch_size=batch_size)
    m = compute_metrics(labels, bert5_preds)
    m.update({"dataset": dataset_name, "model": model_bert5})
    rows.append(m)

    bert3_preds = predict_bert3(texts, batch_size=batch_size)
    m = compute_metrics(labels, bert3_preds)
    m.update({"dataset": dataset_name, "model": model_bert3})
    rows.append(m)

    bertweet3_preds = predict_bertweet3(texts, batch_size=batch_size)
    m = compute_metrics(labels, bertweet3_preds)
    m.update({"dataset": dataset_name, "model": model_bertweet3})
    rows.append(m)

    roberta3_preds = predict_roberta3(texts, batch_size=batch_size)
    m = compute_metrics(labels, roberta3_preds)
    m.update({"dataset": dataset_name, "model": model_roberta3})
    rows.append(m)

    return pd.DataFrame(rows)


def evaluate_roberta_base_reference(df: pd.DataFrame, dataset_name: str, batch_size: int = 64) -> pd.DataFrame:
    df = df.dropna(subset=["text", "label_id"]).copy()
    texts = df["text"].astype(str).tolist()
    labels = df["label_id"].astype(int).tolist()

    preds = predict_roberta_base_reference(texts, batch_size=batch_size)
    m = compute_metrics(labels, preds)
    m.update({"dataset": dataset_name, "model": model_roberta_base_ref})
    return pd.DataFrame([m])

## Phase E - Run the model

In [8]:
dataset_paths = [
    "datasets/tweeteval_sentiment_3class_test.csv",
    "datasets/sst_3class_test.csv",
    "datasets/selfbuilt_database_corrected.csv",
]

batch_size = 64
all_results_main = []
all_results_roberta_base = []

for ds_path in dataset_paths:
    path = Path(ds_path)
    if not path.exists():
        raise FileNotFoundError(f"Dataset file not found: {ds_path}")

    df = pd.read_csv(path)

    print(f"\n=== Main 5 models on: {path.name} | samples={len(df)} ===")
    all_results_main.append(
        evaluate_main_models(
            df,
            dataset_name=path.stem,
            batch_size=batch_size,
            vader_threshold=0.08,
        )
    )

    print(f"=== Separate RoBERTa-base reference on: {path.name} ===")
    all_results_roberta_base.append(
        evaluate_roberta_base_reference(
            df,
            dataset_name=path.stem,
            batch_size=batch_size,
        )
    )

result_main = pd.concat(all_results_main, ignore_index=True)
result_roberta_base_ref = pd.concat(all_results_roberta_base, ignore_index=True)

result_main = result_main[
    [
        "dataset",
        "model",
        "accuracy",
        "f1_macro",
        "precision_macro",
        "recall_macro",
        "f1_negative",
        "f1_neutral",
        "f1_positive",
    ]
]

result_roberta_base_ref = result_roberta_base_ref[
    [
        "dataset",
        "model",
        "accuracy",
        "f1_macro",
        "precision_macro",
        "recall_macro",
        "f1_negative",
        "f1_neutral",
        "f1_positive",
    ]
]

output_dir = Path("results")
output_dir.mkdir(parents=True, exist_ok=True)

csv_main = output_dir / "main_5models_comparison.csv"
csv_roberta_base = output_dir / "roberta_base_reference_comparison.csv"

result_main.to_csv(csv_main, index=False, encoding="utf-8")
result_roberta_base_ref.to_csv(csv_roberta_base, index=False, encoding="utf-8")

print(f"Main 5-model CSV saved to: {csv_main}")
print(f"RoBERTa-base reference CSV saved to: {csv_roberta_base}")

result_main.sort_values(["dataset", "f1_macro"], ascending=[True, False])


=== Main 5 models on: tweeteval_sentiment_3class_test.csv | samples=5990 ===


/opt/anaconda3/envs/COMP6713/lib/python3.10/site-packages/huggingface_hub/file_download.py:949: FutureWarning: `resume_download` is deprecated and will be removed in version 1.0.0. Downloads always resume when possible. If you want to force a new download, use `force_download=True`.
  warnings.warn(
BERT-5class inference: 100%|██████████| 94/94 [04:13<00:00,  2.70s/it]
/opt/anaconda3/envs/COMP6713/lib/python3.10/site-packages/huggingface_hub/file_download.py:949: FutureWarning: `resume_download` is deprecated and will be removed in version 1.0.0. Downloads always resume when possible. If you want to force a new download, use `force_download=True`.
  warnings.warn(
BERT-3class inference: 100%|██████████| 94/94 [04:32<00:00,  2.89s/it]
/opt/anaconda3/envs/COMP6713/lib/python3.10/site-packages/huggingface_hub/file_download.py:949: FutureWarning: `resume_download` is deprecated and will be removed in version 1.0.0. Downloads always resume when possible. If you want to force a new download,

=== Separate RoBERTa-base reference on: tweeteval_sentiment_3class_test.csv ===


Some weights of RobertaForSequenceClassification were not initialized from the model checkpoint at roberta-base and are newly initialized: ['classifier.dense.bias', 'classifier.dense.weight', 'classifier.out_proj.bias', 'classifier.out_proj.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.
RoBERTa-base reference:  11%|█         | 10/94 [00:28<04:03,  2.90s/it]RoBERTa-base reference: 100%|██████████| 94/94 [04:29<00:00,  2.87s/it]
/opt/anaconda3/envs/COMP6713/lib/python3.10/site-packages/huggingface_hub/file_download.py:949: FutureWarning: `resume_download` is deprecated and will be removed in version 1.0.0. Downloads always resume when possible. If you want to force a new download, use `force_download=True`.
  warnings.warn(



=== Main 5 models on: sst_3class_test.csv | samples=1185 ===


...

BERT-5class inference: 100%|██████████| 4/4 [00:11<00:00,  2.94s/it]
/opt/anaconda3/envs/COMP6713/lib/python3.10/site-packages/huggingface_hub/file_download.py:949: FutureWarning: `resume_download` is deprecated and will be removed in version 1.0.0. Downloads always resume when possible. If you want to force a new download, use `force_download=True`.
  warnings.warn(
BERT-3class inference: 100%|██████████| 4/4 [00:12<00:00,  3.01s/it]
/opt/anaconda3/envs/COMP6713/lib/python3.10/site-packages/huggingface_hub/file_download.py:949: FutureWarning: `resume_download` is deprecated and will be removed in version 1.0.0. Downloads always resume when possible. If you want to force a new download, use `force_download=True`.
  warnings.warn(
emoji is not installed, thus not converting emoticons or emojis into text. Install emoji: pip3 install emoji==0.6.0
BERTweet-3class inference: 100%|██████████| 4/4 [00:13<00:00,  3.39s/it]
/opt/anaconda3/envs/COMP6713/lib/python3.10/site-packages/huggingface_h

=== Separate RoBERTa-base reference on: selfbuilt_database_corrected.csv ===


Some weights of RobertaForSequenceClassification were not initialized from the model checkpoint at roberta-base and are newly initialized: ['classifier.dense.bias', 'classifier.dense.weight', 'classifier.out_proj.bias', 'classifier.out_proj.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.
RoBERTa-base reference: 100%|██████████| 4/4 [00:09<00:00,  2.37s/it]

Main 5-model CSV saved to: results/main_5models_comparison.csv
RoBERTa-base reference CSV saved to: results/roberta_base_reference_comparison.csv


,dataset,model,accuracy,f1_macro,precision_macro,recall_macro,f1_negative,f1_neutral,f1_positive
14,selfbuilt_database_corrected,"RoBERTa-3class (pretrained, direct call)",0.825000,0.823175,0.829147,0.825198,0.806452,0.781955,0.881119
13,selfbuilt_database_corrected,"BERTweet-3class (pretrained, direct call)",0.805000,0.801041,0.813072,0.803399,0.762712,0.750000,0.890411
12,selfbuilt_database_corrected,"BERT-3class (pretrained, direct call)",0.735000,0.728555,0.744023,0.732310,0.666667,0.681159,0.837838
11,selfbuilt_database_corrected,"BERT-5class (pretrained, direct call)",0.655000,0.632582,0.671913,0.639967,0.720588,0.483516,0.693642
10,selfbuilt_database_corrected,VADER (baseline),0.500000,0.486326,0.514112,0.495004,0.396040,0.447552,0.615385
9,sst_3class_test,"RoBERTa-3class (pretrained, direct call)",0.669198,0.629830,0.645082,0.632072,0.745798,0.375691,0.768000
8,sst_3class_test,"BERTweet-3class (pretrained, direct call)",0.654008,0.627276,0.649702,0.635519,0.680288,0.407166,0.794372
6,sst_3class_test,"BERT-5class (pretrained, direct call)",0.634599,0.588125,0.593880,0.588684,0.686183,0.345382,0.732809
7,sst_3class_test,"BERT-3class (pretrained, direct call)",0.594937,0.576092,0.617730,0.583644,0.633461,0.359584,0.735230
5,sst_3class_test,VADER (baseline),0.520675,0.470565,0.483365,0.475614,0.498721,0.265010,0.647964


Although BERT-3 does very well on the TweetEval test set, it is overall weaker than RoBERTa-3class and BERTweet-3class on SST and our self-built data. This may be because BERT-3 was fine-tuned on TweetEval (tweet_eval): on that test set the examples are similar to what the model was trained on, so the scores can look high.

Overall, across the three evaluation datasets, the RoBERTa-based pretrained sentiment classifiers achieved more stable performance among the compared approaches.

Therefore, we choose RoBERTa-base as the backbone for our group project and fine-tune it on our data, followed by further extensions.

## Phase F - RoBERTa-base reference (separate)

This section reports RoBERTa-base without task-specific training as an untrained reference point for our later fine-tuning and extension experiments, helping us measure how much improvement comes from our own training, hyperparameter tuning, and model extensions.

In [9]:
result_roberta_base_ref.sort_values(["dataset", "f1_macro"], ascending=[True, False])

,dataset,model,accuracy,f1_macro,precision_macro,recall_macro,f1_negative,f1_neutral,f1_positive
2,selfbuilt_database_corrected,"RoBERTa-base (no fine-tuning, reference)",0.175000,0.166080,0.176864,0.178256,0.107527,0.220779,0.169935
1,sst_3class_test,"RoBERTa-base (no fine-tuning, reference)",0.418565,0.196708,0.139522,0.333333,0.000000,0.000000,0.590125
0,tweeteval_sentiment_3class_test,"RoBERTa-base (no fine-tuning, reference)",0.351252,0.173297,0.117084,0.333333,0.000000,0.000000,0.519891


## Phase G - Export results

In [11]:
from pathlib import Path
output_dir = Path("results")
output_dir.mkdir(parents=True, exist_ok=True)
csv_path = output_dir / "baseline_and_pretrained_model_comparison.csv"

model_order = [
    "VADER (baseline)",
    "BERT-5class (pretrained, direct call)",
    "BERT-3class (pretrained, direct call)",
    "BERTweet-3class (pretrained, direct call)",
    "RoBERTa-3class (pretrained, direct call)",
]

result_export = result_main[result_main["model"].isin(model_order)].copy()
result_export["model"] = pd.Categorical(result_export["model"], categories=model_order, ordered=True)
result_export = result_export.sort_values(["dataset", "model"]).reset_index(drop=True)

result_export.to_csv(csv_path, index=False, encoding="utf-8")
print(f"Saved to: {csv_path}")

Saved to: results/baseline_and_pretrained_model_comparison.csv
